# ETTh1 用电/负荷序列预测：从传统基线到 PatchTST

本 notebook 使用 `ETTh1.csv` 做时间序列预测实验。数据是小时级多变量序列，包含 6 个辅助变量和 1 个目标变量 `OT`。整体任务是：给定过去一段时间的历史窗口，预测未来若干小时的 `OT`。

本 notebook 按模型复杂度分为三层：

1. **传统趋势/ARIMA 基线**：用确定性趋势项解释序列的大方向，帮助判断“只用线性趋势”能做到什么程度。
2. **DLinear**：先把序列分解为趋势项和残差/季节项，再用线性层从历史窗口映射到未来窗口。
3. **PatchTST**：把长序列切成 patch，用 Transformer Encoder 建模 patch 之间的依赖，是现代深度时序预测的常见方案。

评价指标：

- `MSE`：均方误差，对较大的预测偏差更敏感。
- `MAE`：平均绝对误差，更直观地反映平均偏差大小。


## 1. 环境与依赖导入

这一段只负责导入后续需要的库。

- `numpy` / `pandas`：数组和表格数据处理。
- `matplotlib`：画预测曲线。
- `statsmodels.ARIMA`：传统时间序列模型，用作基线。
- `sklearn.metrics`：计算 MSE、MAE。
- `warnings.filterwarnings("ignore")`：减少教学 notebook 中的非关键警告输出。

这里暂时不导入 PyTorch；深度学习模型会在对应代码块里导入，方便分段理解。


In [ ]:
"""ETTh1 时序预测实验：公共依赖导入。"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd
from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore")


## 2. 数据读取与变量定义

这一段完成三个动作：

1. 优先从本地 `ETTh1.csv` 读取数据；如果找不到本地文件，则回退到公开 GitHub URL。
2. 把 `date` 列解析成时间索引，并按时间升序排列。
3. 定义预测目标 `target_col = "OT"`，其它列作为辅助特征。

ETTh1 的列含义可以先按机器学习角度理解：

- `date`：时间戳，小时级。
- `HUFL, HULL, MUFL, MULL, LUFL, LULL`：不同负荷/油温相关的辅助变量。
- `OT`：目标变量，通常作为预测目标。

后续模型统一使用：

$$
X_t = [HUFL_t, HULL_t, MUFL_t, MULL_t, LUFL_t, LULL_t]
$$

以及：

$$
y_t = OT_t
$$

注意：时间序列建模不能随机打乱原始时间顺序再划分训练/测试，否则会产生数据泄漏。


In [ ]:
"""Load the ETTh1 dataset for sequence forecasting.

ETTh1.csv contains hourly electricity transformer measurements.
Use OT as the prediction target and the other columns as features.
"""
from pathlib import Path

ett_url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"

# Prefer a local ETTh1.csv if it exists, then fall back to the public URL.
# The search returns as soon as it finds a file, so it does not scan the whole note vault.
def find_local_ett_file(filename="ETTh1.csv"):
    roots = [Path.cwd(), *list(Path.cwd().parents)[:4]]
    direct_patterns = [
        filename,
        f"datasets/{filename}",
        f"99_Attachments (图片、PDF、附件)/datasets/{filename}",
    ]

    for root in roots:
        for pattern in direct_patterns:
            candidate = root / pattern
            if candidate.is_file():
                return candidate

        # A bounded fallback for cases where the notebook is launched from a subfolder.
        for candidate in root.glob(f"*/datasets/{filename}"):
            if candidate.is_file():
                return candidate

    return None

data_source = find_local_ett_file() or ett_url
df = pd.read_csv(data_source, parse_dates=["date"])
df = df.sort_values("date").set_index("date")

target_col = "OT"
feature_cols = [col for col in df.columns if col != target_col]

# Keep X/y names so the later modeling code can stay the same.
X = df[feature_cols]
y = df[target_col]

print(f"data source: {data_source}")
print(f"shape: {df.shape}")
print(f"time range: {df.index.min()} -> {df.index.max()}")
print(f"target: {target_col}")
print(f"features: {feature_cols}")
display(df.head())


## 3. 传统基线：确定性趋势 + ARIMA 形式

这一段不是 DLinear，而是一个传统统计模型基线。它的作用是回答一个问题：

> 如果只用一条线性时间趋势解释 `OT`，能拟合到什么程度？

### 数学形式

`DeterministicProcess(order=1)` 会生成线性趋势特征：

$$
t = 0, 1, 2, \dots, T-1
$$

这里使用的模型可以理解为：

$$
y_t = \beta_0 + \beta_1 t + \epsilon_t
$$

在代码中使用 `ARIMA(order=(0, 0, 0), exog=X_trend)`，等价于“没有 AR、没有差分、没有 MA，仅用外生趋势项做回归”。

### 关键参数

- `constant=True`：加入截距项 $\beta_0$。
- `order=1`：加入一次线性趋势项 $t$。
- `ARIMA(order=(0, 0, 0))`：不使用自回归和滑动平均结构，只做基线回归。
- `exog=X_trend_normalized`：外生解释变量，即时间趋势。

### 这个基线的局限

它只能学习全局线性趋势，不能建模日周期、周周期、突变和多变量关系。因此它适合作为“最低基线”，不适合作为最终预测模型。


In [ ]:
"""Traditional trend-regression baseline using ARIMA(order=(0, 0, 0)).

This cell is only a simple statistical baseline. It is not DLinear.
"""
from statsmodels.tsa.deterministic import DeterministicProcess

# Build deterministic trend features aligned with df.index.
trend_dp = DeterministicProcess(
    index=df.index,
    constant=True,   # intercept term
    order=1,         # linear time trend
    seasonal=False,  # no calendar-seasonal dummies in this simple baseline
    drop=True,
)

X_trend = trend_dp.in_sample()

# Normalize y and trend features so the fitted coefficients are numerically stable.
y_mean = y.mean()
y_std = y.std()
y_normalized = (y - y_mean) / y_std
X_trend_normalized = (X_trend - X_trend.mean()) / X_trend.std()
X_trend_normalized = X_trend_normalized.fillna(0.0)

# ARIMA(0, 0, 0) with exog is equivalent to regression on exogenous features.
model_trend = ARIMA(endog=y_normalized, exog=X_trend_normalized, order=(0, 0, 0))
result_trend = model_trend.fit()
print(result_trend.summary())

# Convert fitted values back to the original OT scale before plotting.
trend_fitted = result_trend.fittedvalues * y_std + y_mean

plt.figure(figsize=(12, 6))
plt.plot(df.index, y, label="Original OT", color="blue", alpha=0.5)
plt.plot(df.index, trend_fitted, label="Fitted Linear Trend", color="red")
plt.title("ETTh1 OT and Linear-Trend Baseline")
plt.xlabel("Date")
plt.ylabel("OT")
plt.legend()
plt.show()


## 4. DLinear：数据窗口、标准化与 DataLoader

DLinear 的输入不是单个时间点，而是一个历史窗口。

设：

- `seq_len = L`：输入历史长度，这里是过去 336 小时。
- `pred_len = P`：预测未来长度，这里是未来 42 小时。
- `n_features = C`：输入变量数，这里是 7 个变量，即 6 个辅助变量加 `OT`。

每个样本的形状是：

$$
x \in \mathbb{R}^{L \times C}
$$

标签是未来 `OT`：

$$
y \in \mathbb{R}^{P}
$$

### 窗口构造逻辑

如果当前窗口从第 `i` 个点开始，则：

```text
输入 X_i: values[i : i + seq_len]
标签 y_i: OT[i + seq_len : i + seq_len + pred_len]
```

这保证模型只能看到过去，不能看到未来。

### 标准化逻辑

`StandardScaler` 只在训练段上 `fit`，再用于全部数据的 `transform`。这是为了避免测试集信息泄漏。

### 关键参数

- `seq_len`：越大，模型能看到更长周期；但参数更多，训练更慢。
- `pred_len`：预测越长，任务越难。
- `test_len`：最后多少个时间点作为测试区间。
- `batch_size`：每次梯度更新使用多少个窗口样本。
- `epochs`：训练轮数。
- `lr`：学习率。


In [ ]:
"""Prepare sliding-window data for multivariate DLinear forecasting."""
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# Core forecasting settings.
seq_len = 336      # Use the past 336 hours as the input window.
pred_len = 42      # Forecast the next 42 hours.
test_len = 42      # Hold out the final 42 hours for testing.
batch_size = 64
epochs = 30
lr = 1e-3

# Multivariate input: all ETTh1 columns, including the target history.
input_cols = feature_cols + [target_col]
train_end = len(df) - test_len

x_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fit scalers on the training period only to avoid data leakage.
x_scaler.fit(df[input_cols].iloc[:train_end])
y_scaler.fit(df[[target_col]].iloc[:train_end])

X_all = x_scaler.transform(df[input_cols]).astype(np.float32)
y_all = y_scaler.transform(df[[target_col]]).astype(np.float32).ravel()

def make_windows(X, y, seq_len, pred_len):
    """Convert a continuous time series into supervised learning windows.

    Parameters
    ----------
    X : array, shape [time, n_features]
        Standardized multivariate history.
    y : array, shape [time]
        Standardized target series.
    seq_len : int
        Historical lookback length.
    pred_len : int
        Forecast horizon length.
    """
    Xs, ys = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        Xs.append(X[i:i + seq_len])
        ys.append(y[i + seq_len:i + seq_len + pred_len])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_train, y_train = make_windows(
    X_all[:train_end],
    y_all[:train_end],
    seq_len,
    pred_len,
)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=batch_size,
    shuffle=True,
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


## 5. DLinear：数学原理、类与方法

DLinear 的全称可以理解为 **Decomposition Linear**。它的核心假设是：

$$
x_t = trend_t + seasonal_t
$$

其中：

- `trend`：由移动平均得到，表示慢变化趋势。
- `seasonal`：原始序列减去趋势项，表示短期波动/残差。

移动平均可以写成：

$$
trend_t = MA_k(x_t)
$$

残差项：

$$
seasonal_t = x_t - trend_t
$$

然后分别用两个线性层预测未来：

$$
\hat{y} = W_t \cdot vec(trend) + W_s \cdot vec(seasonal)
$$

### 类说明

`MovingAvg`

- `__init__(kernel_size)`：设置移动平均窗口长度。
- `forward(x)`：输入 `[B, L, C]`，输出同形状的趋势项。
- 内部使用 `AvgPool1d`，所以要把张量从 `[B, L, C]` 临时换成 `[B, C, L]`。

`DLinearMultiVar`

- `__init__(seq_len, pred_len, n_features, kernel_size)`：定义趋势分支和残差分支。
- `forward(x)`：先分解，再展平，再线性映射，最后两个分支相加。

### 关键参数

- `kernel_size`：移动平均窗口。太小会保留噪声，太大会过度平滑。
- `in_dim = seq_len * n_features`：线性层输入维度。
- `pred_len`：线性层输出维度，也就是未来预测步数。


In [ ]:
class MovingAvg(nn.Module):
    """Moving average block used to extract the trend component."""

    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1)

    def forward(self, x):
        # x: [B, L, C], where B=batch, L=seq_len, C=n_features
        pad = (self.kernel_size - 1) // 2

        # Replicate boundary values so AvgPool1d keeps roughly the same length.
        front = x[:, 0:1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        x = torch.cat([front, x, end], dim=1)

        # AvgPool1d expects [B, C, L].
        x = x.permute(0, 2, 1)
        x = self.avg(x)
        x = x.permute(0, 2, 1)
        return x


class DLinearMultiVar(nn.Module):
    """Simplified multivariate DLinear that forecasts only the OT target."""

    def __init__(self, seq_len, pred_len, n_features, kernel_size=25):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

        # Both branches receive flattened [seq_len, n_features] history.
        in_dim = seq_len * n_features
        self.trend_linear = nn.Linear(in_dim, pred_len)
        self.seasonal_linear = nn.Linear(in_dim, pred_len)

    def forward(self, x):
        # x: [B, L, C]
        trend = self.moving_avg(x)
        seasonal = x - trend

        # Flatten time and feature dimensions before the linear projection.
        trend = trend.reshape(trend.size(0), -1)
        seasonal = seasonal.reshape(seasonal.size(0), -1)

        return self.trend_linear(trend) + self.seasonal_linear(seasonal)


## 6. DLinear：训练循环

训练过程是标准 PyTorch 回归流程：

1. `model.train()`：进入训练模式。
2. 从 `train_loader` 取一个 batch。
3. `pred = model(xb)`：前向传播得到未来 42 小时预测。
4. `loss = MSE(pred, yb)`：计算均方误差。
5. `loss.backward()`：反向传播。
6. `optimizer.step()`：更新参数。

损失函数：

$$
L = \frac{1}{B P} \sum_{b=1}^{B} \sum_{p=1}^{P}(\hat{y}_{b,p} - y_{b,p})^2
$$

其中 `B` 是 batch size，`P` 是 `pred_len`。

`device` 会优先使用 GPU；如果没有 CUDA，就自动使用 CPU。


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DLinearMultiVar(
    seq_len=seq_len,
    pred_len=pred_len,
    n_features=len(input_cols),
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    model.train()
    losses = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:02d}, Loss: {np.mean(losses):.6f}")


## 7. DLinear：测试集预测、反标准化与指标

这里使用训练集末尾的 `seq_len` 个点作为输入，预测最后 `pred_len` 个点。

输入形状：

$$
X_{test} \in \mathbb{R}^{1 \times seq\_len \times n\_features}
$$

模型输出的是标准化空间中的预测值，所以需要：

```python
y_scaler.inverse_transform(...)
```

还原到原始 `OT` 尺度后再计算 MSE 和 MAE。


In [ ]:
# Use the final seq_len points before the test horizon as model input.
X_test = X_all[train_end - seq_len:train_end]
X_test = torch.tensor(X_test[None, :, :], dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    y_pred_norm = model(X_test).cpu().numpy().reshape(-1, 1)

y_pred = y_scaler.inverse_transform(y_pred_norm).ravel()
y_true = df[target_col].iloc[train_end:train_end + pred_len].values

mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)

print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")


## 8. DLinear：预测曲线可视化

图中蓝线是真实 `OT`，红线是模型预测值。

如果预测曲线很平滑，通常说明 DLinear 正在捕捉趋势，但没有很好捕捉短期波动。这是线性模型的常见现象，不一定是代码错误。


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.index[train_end:train_end + pred_len], y_true, label="True OT", color="blue")
plt.plot(df.index[train_end:train_end + pred_len], y_pred, label="DLinear Predicted OT", color="red")
plt.title("Multivariate DLinear Forecast on ETTh1")
plt.xlabel("Date")
plt.ylabel("OT")
plt.legend()
plt.show()


## 9. PatchTST：模型思想

PatchTST 的核心思想是：不要把每一个时间点都当成一个 token，而是把连续时间点切成短片段 `patch`，再把 patch 当成 Transformer 的 token。

例如：

```text
原始窗口长度 seq_len = 336
patch_len = 16
stride = 8
```

则每个变量会被切成多个重叠 patch。每个 patch 长度为 16，相邻 patch 间隔为 8。

### 数学逻辑

对单个变量的历史序列：

$$
x \in \mathbb{R}^{L}
$$

切成 patch 后：

$$
P \in \mathbb{R}^{N \times patch\_len}
$$

每个 patch 经过线性映射：

$$
z_i = W_p P_i + b_p
$$

再加位置编码：

$$
z_i = z_i + pos_i
$$

最后输入 Transformer Encoder：

$$
H = TransformerEncoder(z)
$$

输出头把所有 patch 表示展平，映射到未来预测窗口：

$$
\hat{y} = W_o \cdot vec(H)
$$

### Channel-independent 设计

这里实现的是简化版 **channel-independent PatchTST**：每个变量单独切 patch、单独经过同一套 Transformer 权重，然后输出每个变量自己的未来序列。

优点：

- 参数量较小。
- 对多变量序列稳定。
- 容易理解和调试。

局限：

- 变量之间没有显式 cross-channel attention。
- 如果变量间关系很强，完整版或 iTransformer 可能更合适。


## 10. PatchTST：数据准备与超参数

这一段和 DLinear 类似，也要做：

1. 标准化。
2. 滑动窗口构造。
3. 按时间顺序划分训练集和验证集。
4. 构造 `DataLoader`。

不同点是：PatchTST 预测的是所有变量的未来值，最后只抽取 `OT` 通道计算指标。

### 关键参数

- `patch_seq_len`：历史窗口长度。
- `patch_pred_len`：预测长度。
- `patch_len`：每个 patch 包含多少个连续时间点。
- `patch_stride`：相邻 patch 的滑动步长。
- `patch_d_model`：Transformer 内部表示维度。
- `patch_n_heads`：多头注意力头数。
- `patch_num_layers`：Transformer Encoder 层数。
- `patch_d_ff`：前馈网络隐藏层维度。
- `patch_dropout`：Dropout 比例，用于抑制过拟合。


In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# PatchTST hyperparameters. Increase epochs/d_model after this runs correctly.
patch_seq_len = 336
patch_pred_len = 42
patch_len = 16
patch_stride = 8
patch_batch_size = 64
patch_epochs = 20
patch_lr = 1e-3
patch_d_model = 64
patch_n_heads = 4
patch_num_layers = 2
patch_d_ff = 128
patch_dropout = 0.1

patch_input_cols = feature_cols + [target_col]
patch_target_idx = patch_input_cols.index(target_col)
patch_test_len = patch_pred_len
patch_train_end = len(df) - patch_test_len

patch_scaler = StandardScaler()
patch_scaler.fit(df[patch_input_cols].iloc[:patch_train_end])
patch_values = patch_scaler.transform(df[patch_input_cols]).astype(np.float32)

def make_multivar_windows(values, seq_len, pred_len, end_idx):
    """Build [history window, future window] pairs for multivariate forecasting."""
    Xs, ys = [], []
    stop = end_idx - seq_len - pred_len + 1
    for i in range(stop):
        Xs.append(values[i:i + seq_len])
        ys.append(values[i + seq_len:i + seq_len + pred_len])
    return np.stack(Xs).astype(np.float32), np.stack(ys).astype(np.float32)

X_patch, y_patch = make_multivar_windows(
    patch_values,
    seq_len=patch_seq_len,
    pred_len=patch_pred_len,
    end_idx=patch_train_end,
)

# Chronological train/validation split inside the training period.
n_val = max(1, int(len(X_patch) * 0.1))
X_patch_train, X_patch_val = X_patch[:-n_val], X_patch[-n_val:]
y_patch_train, y_patch_val = y_patch[:-n_val], y_patch[-n_val:]

patch_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_patch_train), torch.from_numpy(y_patch_train)),
    batch_size=patch_batch_size,
    shuffle=True,
)
patch_val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_patch_val), torch.from_numpy(y_patch_val)),
    batch_size=patch_batch_size,
    shuffle=False,
)

print(f"train windows: {X_patch_train.shape}, val windows: {X_patch_val.shape}")
print(f"target channel: {target_col}, target index: {patch_target_idx}")


## 11. PatchTST：类、方法与张量形状

`ChannelIndependentPatchTST` 是本 notebook 中的简化实现。

### `__init__` 中的模块

- `patch_embedding = nn.Linear(patch_len, d_model)`  
  把每个 patch 从长度 `patch_len` 的原始片段映射到 `d_model` 维向量。

- `position_embedding`  
  给每个 patch 加位置编码，让模型知道 patch 的先后顺序。

- `nn.TransformerEncoderLayer`  
  Transformer 的基本编码层，包含多头自注意力和前馈网络。

- `nn.TransformerEncoder`  
  堆叠多个 Encoder Layer。

- `head = nn.Linear(n_patches * d_model, pred_len)`  
  把所有 patch 的表示展平，再映射到未来 `pred_len` 个点。

### `forward(x)` 的张量流

输入：

```text
x: [batch, seq_len, channels]
```

转置：

```text
[batch, channels, seq_len]
```

切 patch：

```text
[batch, channels, n_patches, patch_len]
```

把变量维度并入 batch：

```text
[batch * channels, n_patches, patch_len]
```

经过 embedding + Transformer：

```text
[batch * channels, n_patches, d_model]
```

输出：

```text
[batch, pred_len, channels]
```


In [ ]:
class ChannelIndependentPatchTST(nn.Module):
    """Compact channel-independent PatchTST implementation."""

    def __init__(
        self,
        seq_len,
        pred_len,
        n_channels,
        patch_len=16,
        stride=8,
        d_model=64,
        n_heads=4,
        num_layers=2,
        d_ff=128,
        dropout=0.1,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.n_channels = n_channels
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1

        self.patch_embedding = nn.Linear(patch_len, d_model)
        self.position_embedding = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(self.n_patches * d_model, pred_len)

        nn.init.normal_(self.position_embedding, std=0.02)

    def forward(self, x):
        # x: [batch, seq_len, channels]
        batch_size, _, n_channels = x.shape
        x = x.permute(0, 2, 1)  # [batch, channels, seq_len]

        # unfold cuts each channel sequence into overlapping patches.
        patches = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        # patches: [batch, channels, n_patches, patch_len]
        patches = patches.reshape(batch_size * n_channels, self.n_patches, self.patch_len)

        z = self.patch_embedding(patches)
        z = z + self.position_embedding
        z = self.dropout(z)
        z = self.encoder(z)
        z = z.reshape(batch_size * n_channels, -1)

        out = self.head(z)  # [batch * channels, pred_len]
        out = out.reshape(batch_size, n_channels, self.pred_len)
        return out.permute(0, 2, 1)  # [batch, pred_len, channels]


## 12. PatchTST：训练逻辑

PatchTST 的训练流程仍然是 PyTorch 标准流程，但有几个额外点：

- 使用 `AdamW`，它在 Adam 的基础上加入权重衰减，更适合 Transformer。
- 使用 `clip_grad_norm_` 限制梯度范数，避免 Transformer 训练初期梯度过大。
- 使用验证集 `val_loss` 保存最佳模型参数。
- 使用简单 early stopping：如果连续若干轮验证损失不下降，就提前停止。

当前代码默认训练所有变量：

$$
loss = MSE(\hat{Y}, Y)
$$

如果你只关心 `OT`，可以改成只对 `OT` 通道计算损失：

```python
loss = criterion(pred[:, :, patch_target_idx], yb[:, :, patch_target_idx])
```


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

patch_model = ChannelIndependentPatchTST(
    seq_len=patch_seq_len,
    pred_len=patch_pred_len,
    n_channels=len(patch_input_cols),
    patch_len=patch_len,
    stride=patch_stride,
    d_model=patch_d_model,
    n_heads=patch_n_heads,
    num_layers=patch_num_layers,
    d_ff=patch_d_ff,
    dropout=patch_dropout,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(patch_model.parameters(), lr=patch_lr, weight_decay=1e-4)

best_state = None
best_val_loss = float("inf")
patience = 5
bad_epochs = 0

for epoch in range(1, patch_epochs + 1):
    patch_model.train()
    train_losses = []

    for xb, yb in patch_train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = patch_model(xb)

        # Train all channels. If you only care about OT, replace this with:
        # loss = criterion(pred[:, :, patch_target_idx], yb[:, :, patch_target_idx])
        loss = criterion(pred, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(patch_model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

    patch_model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in patch_val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = patch_model(xb)
            val_loss = criterion(pred, yb)
            val_losses.append(val_loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(patch_model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | train loss: {train_loss:.6f} | val loss: {val_loss:.6f}")

    if bad_epochs >= patience:
        print(f"Early stopping at epoch {epoch}; best val loss: {best_val_loss:.6f}")
        break

if best_state is not None:
    patch_model.load_state_dict(best_state)


## 13. PatchTST：预测、反标准化与可视化

这一段和 DLinear 的测试逻辑一致：

1. 取测试区间之前的 `patch_seq_len` 个点作为输入。
2. 模型输出标准化空间中的未来所有变量。
3. 使用 `patch_scaler.inverse_transform` 还原到原始尺度。
4. 抽取 `OT` 通道。
5. 计算 MSE/MAE 并画图。

输出形状：

```text
patch_pred_norm: [pred_len, channels]
```

最终用于评价的是：

```text
patch_y_pred: [pred_len]
patch_y_true: [pred_len]
```


In [ ]:
X_patch_test = patch_values[patch_train_end - patch_seq_len:patch_train_end]
X_patch_test = torch.tensor(X_patch_test[None, :, :], dtype=torch.float32).to(device)

patch_model.eval()
with torch.no_grad():
    patch_pred_norm = patch_model(X_patch_test).cpu().numpy()[0]  # [pred_len, channels]

patch_pred_raw = patch_scaler.inverse_transform(patch_pred_norm)
patch_true_raw = df[patch_input_cols].iloc[patch_train_end:patch_train_end + patch_pred_len].values

patch_y_pred = patch_pred_raw[:, patch_target_idx]
patch_y_true = patch_true_raw[:, patch_target_idx]

patch_mse = mean_squared_error(patch_y_true, patch_y_pred)
patch_mae = mean_absolute_error(patch_y_true, patch_y_pred)

print(f"PatchTST MSE: {patch_mse:.4f}")
print(f"PatchTST MAE: {patch_mae:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(df.index[patch_train_end:patch_train_end + patch_pred_len], patch_y_true, label="True OT", color="blue")
plt.plot(df.index[patch_train_end:patch_train_end + patch_pred_len], patch_y_pred, label="PatchTST Predicted OT", color="red")
plt.title("PatchTST Forecast on ETTh1")
plt.xlabel("Date")
plt.ylabel("OT")
plt.legend()
plt.show()


## 14. iTimeSeriesTransformer / iTransformer：模型思想

你说的 `iTimeSeriesTransformer` 在当前环境里不是一个已安装的库类名；这里实现的是 **iTransformer / Inverted Time-Series Transformer** 的简化 PyTorch 版本，并把类命名为 `ITimeSeriesTransformer`，方便在 notebook 里直接使用。

官方 iTransformer 的核心思想是：**把变量当 token，而不是把时间点当 token**。官方仓库说明中也强调，它把独立时间序列视为 variate tokens，用 attention 捕捉多变量相关性。

### 和 PatchTST 的区别

| 模型 | token 是什么 | 主要建模对象 |
|---|---|---|
| PatchTST | 时间 patch | 同一变量内部的长程时间依赖 |
| iTransformer | 变量 channel | 不同变量之间的相关性 |

对于 ETTh1，输入窗口形状是：

```text
[batch, seq_len, channels]
```

iTransformer 会先转成：

```text
[batch, channels, seq_len]
```

然后把每个变量的整段历史 `seq_len` 映射成一个 `d_model` 维 token：

$$
z_c = W x_c + b
$$

其中 `c` 表示变量通道。再通过 Transformer Encoder 在变量 token 之间做 attention：

$$
Attention(Q,K,V) = softmax \left( \frac{QK^T}{\sqrt{d}} \right)V
$$

最后每个变量 token 通过输出头预测未来 `pred_len` 个点。


## 15. iTimeSeriesTransformer：数据准备与关键参数

这一段和 PatchTST 类似，都是多变量预测：

```text
输入: 过去 its_seq_len 小时的所有变量
输出: 未来 its_pred_len 小时的所有变量
评价: 只抽取 OT 通道计算 MSE/MAE
```

### 关键参数

- `its_seq_len`：历史窗口长度。
- `its_pred_len`：预测步长。
- `its_d_model`：每个变量 token 的嵌入维度。
- `its_n_heads`：多头注意力头数。
- `its_num_layers`：Transformer Encoder 层数。
- `its_d_ff`：前馈网络隐藏层维度。
- `its_dropout`：Dropout 比例。

### 适用场景

iTransformer 更适合变量之间存在较强相关性的多变量序列。如果只用单变量 `OT`，它的优势会下降；因此这里默认用 7 个 ETTh1 变量一起训练。


In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

its_seq_len = 336
its_pred_len = 42
its_batch_size = 64
its_epochs = 20
its_lr = 1e-3
its_d_model = 64
its_n_heads = 4
its_num_layers = 2
its_d_ff = 128
its_dropout = 0.1

its_input_cols = feature_cols + [target_col]
its_target_idx = its_input_cols.index(target_col)
its_test_len = its_pred_len
its_train_end = len(df) - its_test_len

its_scaler = StandardScaler()
its_scaler.fit(df[its_input_cols].iloc[:its_train_end])
its_values = its_scaler.transform(df[its_input_cols]).astype(np.float32)

def make_its_windows(values, seq_len, pred_len, end_idx):
    """Build supervised windows for multivariate iTransformer forecasting."""
    Xs, ys = [], []
    stop = end_idx - seq_len - pred_len + 1
    for i in range(stop):
        Xs.append(values[i:i + seq_len])
        ys.append(values[i + seq_len:i + seq_len + pred_len])
    return np.stack(Xs).astype(np.float32), np.stack(ys).astype(np.float32)

X_its, y_its = make_its_windows(
    its_values,
    seq_len=its_seq_len,
    pred_len=its_pred_len,
    end_idx=its_train_end,
)

# Chronological validation split.
its_n_val = max(1, int(len(X_its) * 0.1))
X_its_train, X_its_val = X_its[:-its_n_val], X_its[-its_n_val:]
y_its_train, y_its_val = y_its[:-its_n_val], y_its[-its_n_val:]

its_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_its_train), torch.from_numpy(y_its_train)),
    batch_size=its_batch_size,
    shuffle=True,
)
its_val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_its_val), torch.from_numpy(y_its_val)),
    batch_size=its_batch_size,
    shuffle=False,
)

print(f"train windows: {X_its_train.shape}, val windows: {X_its_val.shape}")
print(f"target channel: {target_col}, target index: {its_target_idx}")


## 16. iTimeSeriesTransformer：类、方法与张量流

`ITimeSeriesTransformer` 是一个紧凑版 inverted Transformer。

### `__init__` 里的核心模块

- `value_embedding = nn.Linear(seq_len, d_model)`  
  把单个变量的完整历史窗口映射成一个 token。

- `channel_embedding`  
  给不同变量加入可学习的位置/通道编码。

- `TransformerEncoderLayer`  
  在变量 token 之间做 self-attention。

- `head = nn.Linear(d_model, pred_len)`  
  把每个变量 token 映射成未来 `pred_len` 个点。

### `forward(x)` 张量变化

```text
x: [batch, seq_len, channels]
```

转置为变量 token：

```text
[batch, channels, seq_len]
```

线性嵌入：

```text
[batch, channels, d_model]
```

Transformer 编码：

```text
[batch, channels, d_model]
```

输出预测：

```text
[batch, pred_len, channels]
```


In [ ]:
class ITimeSeriesTransformer(nn.Module):
    """Compact iTransformer-style model for multivariate forecasting.

    It treats each variable/channel as a token and uses self-attention across variables.
    """

    def __init__(
        self,
        seq_len,
        pred_len,
        n_channels,
        d_model=64,
        n_heads=4,
        num_layers=2,
        d_ff=128,
        dropout=0.1,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.n_channels = n_channels

        self.value_embedding = nn.Linear(seq_len, d_model)
        self.channel_embedding = nn.Parameter(torch.zeros(1, n_channels, d_model))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, pred_len)

        nn.init.normal_(self.channel_embedding, std=0.02)

    def forward(self, x):
        # x: [batch, seq_len, channels]
        x = x.permute(0, 2, 1)  # [batch, channels, seq_len]

        # Each variable's whole historical window becomes one token.
        z = self.value_embedding(x)  # [batch, channels, d_model]
        z = z + self.channel_embedding
        z = self.dropout(z)

        # Self-attention runs across variable tokens.
        z = self.encoder(z)

        out = self.head(z)  # [batch, channels, pred_len]
        return out.permute(0, 2, 1)  # [batch, pred_len, channels]


## 17. iTimeSeriesTransformer：训练逻辑

训练逻辑与 PatchTST 相同：

1. 输入 `[batch, its_seq_len, channels]`。
2. 输出 `[batch, its_pred_len, channels]`。
3. 默认对所有变量计算 MSE。
4. 保存验证损失最低的模型参数。

如果只想优化 `OT`，同样可以把损失函数改为：

```python
loss = criterion(pred[:, :, its_target_idx], yb[:, :, its_target_idx])
```

注意：iTransformer 的优势来自变量间 attention。如果只对 `OT` 通道计算损失，模型仍然能通过 `OT` 之外的变量 token 建模关系，但训练信号会更集中在目标变量上。


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

its_model = ITimeSeriesTransformer(
    seq_len=its_seq_len,
    pred_len=its_pred_len,
    n_channels=len(its_input_cols),
    d_model=its_d_model,
    n_heads=its_n_heads,
    num_layers=its_num_layers,
    d_ff=its_d_ff,
    dropout=its_dropout,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(its_model.parameters(), lr=its_lr, weight_decay=1e-4)

its_best_state = None
its_best_val_loss = float("inf")
its_patience = 5
its_bad_epochs = 0

for epoch in range(1, its_epochs + 1):
    its_model.train()
    train_losses = []

    for xb, yb in its_train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = its_model(xb)

        # Train all channels. To optimize OT only, use the commented loss below.
        # loss = criterion(pred[:, :, its_target_idx], yb[:, :, its_target_idx])
        loss = criterion(pred, yb)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(its_model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

    its_model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in its_val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = its_model(xb)
            val_loss = criterion(pred, yb)
            val_losses.append(val_loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))

    if val_loss < its_best_val_loss:
        its_best_val_loss = val_loss
        its_best_state = copy.deepcopy(its_model.state_dict())
        its_bad_epochs = 0
    else:
        its_bad_epochs += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | train loss: {train_loss:.6f} | val loss: {val_loss:.6f}")

    if its_bad_epochs >= its_patience:
        print(f"Early stopping at epoch {epoch}; best val loss: {its_best_val_loss:.6f}")
        break

if its_best_state is not None:
    its_model.load_state_dict(its_best_state)


## 18. iTimeSeriesTransformer：预测与可视化

测试阶段流程：

1. 取测试区间之前的 `its_seq_len` 个点。
2. 输出未来 `its_pred_len` 个点的所有变量。
3. 使用 `its_scaler.inverse_transform` 反标准化。
4. 抽取 `OT` 通道。
5. 计算 MSE/MAE，并画出真实值与预测值。

和 PatchTST 对比时，重点看：

- MSE/MAE 是否下降。
- 曲线是否比 DLinear 更能跟随局部波动。
- 是否出现过拟合：训练损失下降，但验证损失不降。


In [ ]:
X_its_test = its_values[its_train_end - its_seq_len:its_train_end]
X_its_test = torch.tensor(X_its_test[None, :, :], dtype=torch.float32).to(device)

its_model.eval()
with torch.no_grad():
    its_pred_norm = its_model(X_its_test).cpu().numpy()[0]  # [pred_len, channels]

its_pred_raw = its_scaler.inverse_transform(its_pred_norm)
its_true_raw = df[its_input_cols].iloc[its_train_end:its_train_end + its_pred_len].values

its_y_pred = its_pred_raw[:, its_target_idx]
its_y_true = its_true_raw[:, its_target_idx]

its_mse = mean_squared_error(its_y_true, its_y_pred)
its_mae = mean_absolute_error(its_y_true, its_y_pred)

print(f"iTimeSeriesTransformer MSE: {its_mse:.4f}")
print(f"iTimeSeriesTransformer MAE: {its_mae:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(df.index[its_train_end:its_train_end + its_pred_len], its_y_true, label="True OT", color="blue")
plt.plot(df.index[its_train_end:its_train_end + its_pred_len], its_y_pred, label="iTimeSeriesTransformer Predicted OT", color="purple")
plt.title("iTimeSeriesTransformer Forecast on ETTh1")
plt.xlabel("Date")
plt.ylabel("OT")
plt.legend()
plt.show()


## 19. 模型对比与下一步

从建模能力看：

| 模型 | 主要能力 | 优点 | 局限 |
|---|---|---|---|
| 趋势/ARIMA 基线 | 线性趋势拟合 | 简单、可解释 | 不能捕捉复杂周期和非线性 |
| DLinear | 趋势/残差分解 + 线性映射 | 快、稳定、强 baseline | 容量有限，预测曲线可能偏平滑 |
| PatchTST | patch + Transformer Encoder | 能建模较复杂的长程时间依赖 | 参数更多，训练更慢，需要调参 |
| iTimeSeriesTransformer | 变量 token + Transformer Encoder | 更强调多变量相关性 | 时间动态主要靠线性 embedding 压缩，简化版能力有限 |

建议实验顺序：

1. 先跑 DLinear，确认窗口、标准化、反标准化流程正确。
2. 再跑 PatchTST，观察长程时间依赖建模是否改善。
3. 再跑 iTimeSeriesTransformer，观察变量间 attention 是否带来收益。
4. 如果只关心 `OT`，可以分别把 PatchTST 和 iTimeSeriesTransformer 的损失改为只计算 `OT` 通道。
5. 如果训练慢，优先减小 `epochs`、`d_model`、`num_layers`；如果欠拟合，再逐步加大。
